# 04 (stretch) Metric learning: a small image adapter for instance matching

**Compute bucket, optional. Needs REAL lost/found pairs in `eval/pairs/manifest.csv` with photos on both sides (50+ pairs; 150+ is better).**

CLIP tells a laptop from a phone well, but it is weaker at "this exact laptop versus another black laptop". This notebook trains a
**residual adapter** on top of frozen CLIP image embeddings with a triplet loss over your pairs:

    image = normalize(x + x @ delta)

Only image embeddings are adapted, so text and text-to-photo similarity keep CLIP's geometry. Output `projection.pt` is picked up by
`backend/app/ml/vision/embedder.py` when it sits in `backend/weights/`. Always rerun `python -m eval.run_eval --pairs real` afterwards
and keep it only if P@1 improves on the held-out pairs.

In [ ]:
!pip -q install open_clip_torch pillow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import csv, json, random
from pathlib import Path
import numpy as np, torch, open_clip
from PIL import Image

PAIRS = Path("/content/drive/MyDrive/reunite/pairs")     # copy eval/pairs here: manifest.csv plus the images it points to
DEST = Path("/content/drive/MyDrive/reunite")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model = model.to(device).eval()
random.seed(0); torch.manual_seed(0)

rows = list(csv.DictReader(open(PAIRS / "manifest.csv")))
by = {}
for r in rows:
    if r["image"]:
        by.setdefault(r["pair_id"], {})[r["side"]] = PAIRS / r["image"]
pairs = {p: s for p, s in by.items() if "lost" in s and "found" in s}
print(len(pairs), "pairs with a photo on both sides")
assert len(pairs) >= 50, "collect more pairs first (see eval/README.md)"

In [ ]:
@torch.no_grad()
def emb(path):
    e = model.encode_image(preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)).float()
    return torch.nn.functional.normalize(e, dim=-1).cpu()[0]

ids = sorted(pairs); random.shuffle(ids)
cut = int(len(ids) * 0.7); train_ids, val_ids = ids[:cut], ids[cut:]      # split by pair id, never by image
E = {p: (emb(pairs[p]["lost"]), emb(pairs[p]["found"])) for p in ids}

def p_at_1(delta, which):
    L = torch.stack([E[p][0] for p in which]); Fd = torch.stack([E[p][1] for p in which])
    adapt = lambda x: torch.nn.functional.normalize(x + x @ delta, dim=-1)
    sims = adapt(L) @ adapt(Fd).T
    return (sims.argmax(1) == torch.arange(len(which))).float().mean().item()

zero = torch.zeros(512, 512)
print("held-out image-only P@1 before:", round(p_at_1(zero, val_ids), 3))

In [ ]:
delta = torch.nn.Parameter(torch.zeros(512, 512))
opt = torch.optim.AdamW([delta], lr=5e-4, weight_decay=0.0)
margin, reg = 0.2, 1e-2
L = torch.stack([E[p][0] for p in train_ids]); Fd = torch.stack([E[p][1] for p in train_ids])
adapt = lambda x: torch.nn.functional.normalize(x + x @ delta, dim=-1)
best, best_delta = p_at_1(zero, val_ids), zero.clone()
for step in range(600):
    a, p = adapt(L), adapt(Fd)
    sims = a @ p.T
    pos = sims.diag().unsqueeze(1)
    neg_mask = ~torch.eye(len(L), dtype=torch.bool)
    loss = torch.relu(margin + sims - pos)[neg_mask].mean() + reg * delta.pow(2).sum()      # triplet loss with all in-batch negatives
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0:
        with torch.no_grad():
            acc = p_at_1(delta.detach(), val_ids)
        if acc > best: best, best_delta = acc, delta.detach().clone()
print("held-out image-only P@1 after:", round(best, 3))

In [ ]:
torch.save({"delta": best_delta.cpu(), "note": "residual adapter: image = normalize(x + x @ delta)"}, DEST / "projection.pt")
json.dump({"heldout_image_p_at_1_before": p_at_1(zero, val_ids), "heldout_image_p_at_1_after": best, "n_train_pairs": len(train_ids), "n_val_pairs": len(val_ids)},
          open(DEST / "projection_metrics.json", "w"), indent=2)
print("saved projection.pt")

Copy `projection.pt` to `backend/weights/`, restart the backend, then **rerun the real-pair eval**. If P@1 does not improve on held-out
pairs, delete the file: the adapter is only worth keeping when it earns its place.